# Storage Mount — ADLS Gen2 OAuth2

Configures Spark to authenticate to ADLS Gen2 via Service Principal stored in Key Vault.

Run once per cluster session, or called via  from other notebooks.

In [ ]:
# Cell 1 — Load all lib/ modules (one line)
%run ./lib/bootstrap.py

In [ ]:
# Cell 2 — Parameters (set by ADF widget or manually)
dbutils.widgets.text("storage_account", "sadataeng260524dev", "Storage Account")
dbutils.widgets.text("kv_scope",        "kv-scope",           "Key Vault Scope")

STORAGE_ACCOUNT = dbutils.widgets.get("storage_account")
KV_SCOPE        = dbutils.widgets.get("kv_scope")

log = setup_logger("storagemount")
log.info(f"Storage account : {STORAGE_ACCOUNT}")
log.info(f"Key Vault scope : {KV_SCOPE}")

In [ ]:
# Cell 3 — Verify secrets + configure OAuth2 + set path variables
try:
    auth   = AdlsAuth(dbutils, spark, scope=KV_SCOPE)
    paths  = auth.setup(STORAGE_ACCOUNT)

    BRONZE = paths["BRONZE"]
    SILVER = paths["SILVER"]
    GOLD   = paths["GOLD"]

    log.info(f"BRONZE : {BRONZE}")
    log.info(f"SILVER : {SILVER}")
    log.info(f"GOLD   : {GOLD}")

except RuntimeError as e:
    log.error(f"Setup failed: {e}")
    raise   # re-raise so ADF / notebook marks the run as failed


In [ ]:
# Cell 4 — Smoke test: verify ADLS access is working
try:
    files = dbutils.fs.ls(f"{BRONZE}/")
    log.info(f"ADLS access confirmed — {len(files)} container(s) visible ✅")
    display(dbutils.fs.ls(f"{BRONZE}/"))
except Exception as e:
    log.error(f"ADLS access failed: {e}")
    log.error("Check: SP has Storage Blob Data Contributor on the storage account")
    raise
